# Практика: рекурсия, lambda, pipeline

**Пара КТП 8** (2 ч) — отработка.

Минимум сдачи: A1–A2, B1–B2, C1. Опора — [пара 7](../07_recursion/lesson.ipynb).

**Данные:** `NESTED_API_RESPONSE`, `CATEGORY_TREE`, `EXAM_SCORES`, `MODEL_RUNS`, `FEATURE_ROWS`, `FEATURE_POINTS`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../..').resolve()))
from data.module_datasets import (
    APARTMENTS, EXAM_SCORES, PREDICTIONS, LABELS,
    NESTED_API_RESPONSE, CATEGORY_TREE, FEATURE_ROWS,
    MODEL_RUNS, FEATURE_POINTS,
    PRICE_INTERCEPT, PRICE_COEF_AREA,
)


---

# A. Рекурсия


## A1. `extract_ids`

In [ ]:
def extract_ids(nested):
    pass


assert extract_ids(NESTED_API_RESPONSE) == [1, 2, 3]


## A2. `walk_categories`

Тот же контракт, что на паре 7: `(name, children)`.

In [ ]:
def walk_categories(node, depth=0):
    pass


# walk_categories(CATEGORY_TREE)


## A3 (углубление). `count_leaves`

In [ ]:
def count_leaves(node):
    pass


# assert count_leaves(CATEGORY_TREE) == 3


---

# B. Lambda


## B1. `filter` — аномалии (свой порог)

In [ ]:
anomalies = list(filter(lambda s: s < 50 or s > 90, EXAM_SCORES))
assert set(anomalies) == {40, 48, 91, 95}
print(anomalies)

## B2. Leaderboard `MODEL_RUNS`

Сортировка: f1 ↓; при равенстве — id ↑.

In [ ]:
leaderboard = sorted(MODEL_RUNS, key=lambda r: (-r['f1'], r['id']))
assert [r['id'] for r in leaderboard] == [25, 30, 305, 101, 200]
print([r['id'] for r in leaderboard])


## B3 (углубление). Farthest point

In [ ]:
# farthest = max(FEATURE_POINTS, key=lambda p: ...)
# assert farthest == [55, 12]
# print(farthest)


---

# C. Pipeline


## C1. `apply_pipeline`

In [ ]:
def apply_pipeline(data, steps):
    result = data
    for step in steps:
        result = step(result)
    return result


## C1. Цепочка для одной квартиры

`dict → area → scale → predict`

In [ ]:
def extract_area(row):
    return row['area_sqm']


def scale_area(area, max_area=60):
    return area / max_area


def predict_from_scaled(scaled_area):
    return PRICE_INTERCEPT + PRICE_COEF_AREA * (scaled_area * 60)


apt_pipeline = [extract_area, scale_area, predict_from_scaled]
pred = apply_pipeline(FEATURE_ROWS[0], apt_pipeline)
assert abs(pred - (PRICE_INTERCEPT + PRICE_COEF_AREA * 28)) < 1e-9
print('predicted mln:', pred)

## Эксперимент: порядок шагов

Что будет, если вызвать scale до extract? Запишите ошибку/вывод.

In [ ]:
# эксперимент


## C2 (углубление). Текстовый pipeline

In [ ]:
text_pipeline = [
    lambda s: s.strip().lower(),
    lambda s: s.split(),
    lambda tokens: [t for t in tokens if len(t) >= 4],
]

assert apply_pipeline('  Data science ML course  ', text_pipeline) == ['data', 'science', 'course']
print(apply_pipeline('  Data science ML course  ', text_pipeline))

## Мост к артефакту

Пары 9–10: [artifact/PROJECT.md](../../artifact/PROJECT.md) и [LESSON пары 9](../09_artifact_build/LESSON.md).

## Итог

**Pipeline = композиция функций.** Дальше — итоговый модуль `text_stats`.